# Testing the CIMA package

Test the loading of the S3-CIMA Pytorch package. Make sure to pip install the package into an environment before running: 

    pip install s3cima

S3-CIMA provides two main functions - the first is the process_csv function, to pre-process input data and filter any unwanted/outlier cell types. The second is the run_s3cima function, to run a s3-cima analysis.

In [1]:
from s3cima.utils.preprocess import process_csv

The process_csv function requires two inputs csvs - a marker csv (containing log-normalised gene expression) and a paired metadata csv. These MUST be paired, aka the cell on the 5th row of the markers csv must be the same as the cell on the 5th row of the metadata csv. Additionally, in the process_csv function, you must specify the column names of some important variables which S3-CIMA uses and that are in the metadata csv : 

- x_col : The 'X' coordinate of cells 
- y_col : The 'Y' coordinate of cells 
- cell_id_col : A unique cell id to identify each cell
- cell_type_col : The annotated cell types of each cell
- sample_id_col : The patient/sample unique id
- image_id_col : A unique id for each image (NOTE - if your dataset has only one image per patient/sample, DO NOT re-use the patient id column - create and assign a unique image id column anyways)
- condition_col : This is the outcome column of interest (disease conditions for a classification for instance)
- filter_cell_types : An optional argument - should be a list of strings to determine whether some cell types should be removed from the csv.

In [2]:
# Load data
intensity, genes, x, y, cell_id, cell_type, sample_id, image_id, labels, label_map = process_csv(
    markers_path      = "/Users/gabrielduval/Documents/S3-CIMA/tests/data/cima_test_markers.csv",
    meta_path         = "/Users/gabrielduval/Documents/S3-CIMA/tests/data/cima_test_meta.csv",
    x_col             = "spatial_dim0",
    y_col             = "spatial_dim1",
    cell_id_col       = "cell_id",
    cell_type_col     = "cell_type",
    sample_id_col     = "sample",
    image_id_col      = "image_id",
    condition_col     = "condition",
    filter_cell_types = ["Schwann_cells", "Muscular_cells"],   # or None to keep all
)

[INFO] Loading metadata from   : /Users/gabrielduval/Documents/S3-CIMA/tests/data/cima_test_meta.csv
[INFO] Loading markers from    : /Users/gabrielduval/Documents/S3-CIMA/tests/data/cima_test_markers.csv
[INFO] Loaded 3195 cells × 23 markers.
[INFO] Marker value range: [0.0000, 7.1317] — consistent with log-normalised data.
[INFO] filter_cell_types validated — 2 type(s) will be removed.
[INFO] All checks passed.
[INFO] Cell type filter applied: 3195 → 3130 cells retained (65 removed).
[INFO] Final output: 3130 cells × 23 markers.


Once this has run, you are ready to run an S3-CIMA analysis.

In [3]:
from s3cima.s3cima import run_s3cima

There are a lot of arguments in the s3cima function that you can modify depending on your need. Documentation on the run_s3cima function will soon be available. Below is a base level example with some simple yet critical parameters for an anchor CD8 cell type :

In [ ]:
save_path = "enter/the/path/to/save/output/here"

run_s3cima(
    "CD8",
    image_id, 
    sample_id, 
    intensity, 
    genes, 
    cell_type, 
    x, 
    y, 
    cell_id, 
    labels,
    label_map,
    K=50,
    ncell = 30,
    random_ctrl=False,
    n_val_folds=3,
    num_workers=24,
    batch_size=128,
    lr = 0.001,
    nruns=10,
    save_path = save_path,
    seed=43
)

Another example this time using the background "BG" instead of a particular anchor - this selects anchor cells randomly instead of focusing on a specific cell type to classify. Can often be a good baseline.

In [ ]:
save_path = "enter/the/path/to/save/output/here"

run_s3cima(
    "BG",
    image_id, 
    sample_id, 
    intensity, 
    genes, 
    cell_type, 
    x, 
    y, 
    cell_id, 
    labels,
    label_map,
    K=50,
    ncell = 30,
    random_ctrl=False,
    n_val_folds=3,
    bg_sets=1000,    # Number of random background sets to draw from each sample/patient
    num_workers=24,
    batch_size=128,
    lr = 0.001,
    nruns=10,
    save_path = save_path,
    seed=43
)